<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #ffffff;
    background-color: #9E1B32;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    Model Evaluation During Extreme Events
</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Savalan Naser Neisary <a href="mailto:snaserneisary@crimson.ua.edu">(snaserneisary@crimson.ua.edu)</a></li>
</ul>


**Last Updated:** 10/10/2026

**Purpose:**

The goal in this notebook is to evaluate the model for different inputs from 2002 to 2022 across the entire time frame, during extreme events using the Streamflow Extreme Event Dataset (SEED) and it performance in capturing streamflow drought events. For more information about SEED please read 
<span style="background-color:rgba(184,34,61,0.12); padding:4px 6px; border-radius:6px;">
  <a href="https://doi.org/10.1111/1752-1688.70093" target="_blank" style="font-weight:bold; color:#E04A63;">
    Model-Agnostic Framework for Evaluating Hydrological Models Under Extreme Events Across the Contiguous United States paper
  </a>
</span>.

**Audience:**

Researchers and practitioners working with NextGen hydrologic simulations. Users should be familiar with Python, Jupyter Notebooks, and basic hydrologic modeling concepts.

**Description:**

<div style="font-size:14px; margin-bottom:0; padding-bottom:0;">

- We first evaluate the NextGen LSTM module predictions forced by two forcing datasets, AORC and IMERG, with NWM, compared to observations using two different forcings from 2002 to 2022.
- Then we evaluate its capability in capturing streamflow drought events.
- Finally, we evaluate the models during extreme drought and flood events. 
</div>
<div style="margin-top:-16px;"></div>

**Data Description:**

We have four daily streamflow datasets for one station in Alabama (USGS 02450250): USGS observations, the NWM v3.0 retrospective, and two NextGen LSTM simulations with two different forcings (AORC and IMERG).

We also use the **Streamflow Extreme Event Dataset (SEED)**, which lists the dates of extreme flood and drought events (with their return periods) at USGS gages across the contiguous United States. It was built to evaluate models during extremes, independent of the model used.

- **Paper:** Alam, M. S., Johnson, R., Naser Neisary, S., Halgren, J., & Burian, S. (2025). Model-Agnostic Framework for Evaluating Hydrological Models Under Extreme Events Across the Contiguous United States. *JAWRA Journal of the American Water Resources Association*. <a href="https://doi.org/10.1111/1752-1688.70093" target="_blank" style="font-weight:bold; text-decoration:underline; color:#9E1B32;">https://doi.org/10.1111/1752-1688.70093</a>
- **SEED GitHub:** <a href="https://github.com/shahab122/SEED-ROSET" target="_blank" style="font-weight:bold; text-decoration:underline; color:#9E1B32;">shahab122/SEED-ROSET</a>
- **CSES:** the Community Streamflow Evaluation System, an open-source tool to evaluate streamflow models against USGS observations, including during SEED events. Repository: <a href="https://github.com/whitelightning450/Community-Streamflow-Evaluation-System" target="_blank" style="font-weight:bold; text-decoration:underline; color:#9E1B32;">Community-Streamflow-Evaluation-System</a>; web app: <a href="https://github.com/AlabamaWaterInstitute/Tethys-CSES" target="_blank" style="font-weight:bold; text-decoration:underline; color:#9E1B32;">Tethys-CSES</a>


<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    1. Prepare the Python Environment
  </h3>
</div>

We prepare the **Python environment** by importing all required libraries and utilities used throughout this TEEHR evaluation workflow, which supports **data processing**, **time-series visualization**, and **NextGen/TEEHR evaluation**.

In [ ]:
from utils.drought_events import (
    load_daily_flows,
    metrics_table,
    run_drought_comparison,
    key_observed_events,
    event_metrics,
    period_metrics,
    plot_metrics_bars,
    plot_period_series,
    plot_drought_events,
    plot_drought_summary,
)

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    2. Model Performance over the Entire Period (2002–2022)
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
Here we compare every model with the USGS observations over the whole record, using daily flow (m³/s) on the days that all datasets have (2002-01-01 to 2022-12-30). The models are the <strong>NWM retrospective</strong> and the <strong>NextGen LSTM runs</strong> (one column per forcing). The daily flows are stored in <code>data/daily_flows.parquet</code>.
</p>

<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
  <strong>Metrics (each model vs. USGS observations):</strong>
  <ul style="margin-top: 8px; margin-bottom: 0;">
    <li><strong>KGE</strong> (Kling-Gupta Efficiency): combines correlation, variability and bias. Best = 1.</li>
    <li><strong>NSE</strong> (Nash-Sutcliffe Efficiency): how well the model matches the observations, mostly driven by high flows. Best = 1; below 0 is worse than using the observed mean.</li>
    <li><strong>NNSE</strong> (Normalized NSE): NSE rescaled to 0–1. Best = 1.</li>
    <li><strong>logNSE</strong>: NSE of the log of flow, which gives more weight to low flows. Best = 1.</li>
    <li><strong>RMSE</strong> (Root Mean Square Error, m³/s): typical size of the error. Best = 0.</li>
    <li><strong>PBIAS</strong> (Percent Bias, %): total volume error. Best = 0; positive means the model is too high.</li>
  </ul>
</div>


In [ ]:
SITE_NO = "02450250"

# Daily flows (m³/s): Observed (USGS), NWM and each NextGen run
flows = load_daily_flows(models=["NWM", "NextGen (AORC)", "NextGen (IMERG)"])
flows.head()

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Metrics for the Entire Period
</h4>
<p style="font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:6px;">
One panel per metric and one bar per model. The dashed line shows the best possible value of each metric (a perfect match with the observations).
</p>


In [ ]:
# Metrics of every model against USGS observations, on the days all models have
all_period_metrics = metrics_table(flows.dropna())
plot_metrics_bars(all_period_metrics, title="Entire period (2002–2022): models vs. USGS observations")

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    3. Daily Flow Comparison and Drought Events (Observed vs. NWM vs. NextGen)
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">

Observed (USGS), NWM and NextGen daily flows (m³/s) are stored in `data/daily_flows.parquet`.
Daily drought thresholds (1980–2025) are read from `data/drought_thresholds.parquet`.
</p>



<div style="background-color:rgba(184,34,61,0.12); border-left: 5px solid #E04A63; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; margin-top: 10px;">
<h4 style="margin-top:0;">Drought events</h4>

A day is in <strong>drought</strong> when the flow is below a daily threshold. The thresholds come from the observed flow over 1980–2025, separately for each calendar day, so a normal dry summer is not called a drought.

<ul style="margin-top: 8px; margin-bottom: 0;">
  <li><strong><code>Q_LEVEL</code></strong> (95, 90, 80 or 70): the threshold. Q80 is the flow exceeded 80% of the time on that calendar day (the 20th percentile). Q95 finds only the most severe droughts; Q70 finds more, milder ones.</li>
  <li><strong><code>PERSISTENCE_DAYS</code></strong>: the minimum number of days in a row below the threshold to count as an event. It removes short dry spells.</li>
  <li><strong><code>GAP_DAYS</code></strong>: two events separated by this many days or fewer are joined into one, so one rainy day does not split a drought.</li>
</ul>

<p style="margin-bottom:0;"><strong>Severity</strong> = mean % below the threshold over the drought days of an event.</p>

<p style="margin-bottom:0;"><strong>Why change them?</strong> There is no single definition of drought. Different studies and uses (water supply, ecology) pick different thresholds and lengths. Changing them shows whether a model's drought skill holds for mild and severe droughts, and for short and long events.</p>
</div>

In [ ]:
# ------------------ Your choices ------------------
Q_LEVEL = 80            # 95, 90, 80 or 70
PERSISTENCE_DAYS = 7    # minimum event length (days)
GAP_DAYS = 3            # events separated by this many days or fewer are joined
# --------------------------------------------------

drought_summary, drought_events, drought_threshold = run_drought_comparison(
    flows,
    site_no=SITE_NO,
    q=Q_LEVEL,
    persistence_days=PERSISTENCE_DAYS,
    gap_days=GAP_DAYS,
)
# Number of events, longest event and most severe event: each model vs. observations
plot_drought_summary(drought_summary)

In [ ]:
# Drought events found in each series (one bar per event, darker = more severe)
plot_drought_events(drought_events)

<div style="background:#9E1B32; border-left:6px solid #F0627A; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#fff5f7;">
    4. Model Performance during SEED Extreme Events
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
The <strong>Streamflow Extreme Event Dataset (SEED)</strong> lists the dates of extreme flood and drought events at USGS gages. For our station, <strong>USGS 02450250 (Sipsey Fork near Grayson, AL; NWM COMID 18578829)</strong>, we use one flood and one drought event from SEED:
</p>

<table style="border-collapse: collapse; font-size: 14px;">
  <thead>
    <tr style="background-color: #ffffff; color: #9E1B32;">
      <th style="padding: 8px; text-align: left; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Event</th>
      <th style="padding: 8px; text-align: left; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Start</th>
      <th style="padding: 8px; text-align: left; color: #9E1B32; border-bottom: 2px solid #9E1B32;">End</th>
      <th style="padding: 8px; text-align: left; color: #9E1B32; border-bottom: 2px solid #9E1B32;">Return period</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 8px;">Flood</td><td style="padding: 8px;">2006-12-29</td><td style="padding: 8px;">2007-01-12</td><td style="padding: 8px;">100 years</td></tr>
    <tr><td style="padding: 8px;">Drought</td><td style="padding: 8px;">2021-07-26</td><td style="padding: 8px;">2021-08-25</td><td style="padding: 8px;">100 years</td></tr>
  </tbody>
</table>

<p style="margin-top:10px;">
We compute the same metrics as in Section 2, separately for each event window. The windows are short (15 and 31 days), so the metrics change a lot with small errors and should be compared between models rather than with the full-period values.
</p>


<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 SEED Drought Event (2021-07-26 to 2021-08-25)
</h4>


In [ ]:
# Daily streamflow during the SEED drought event
plot_period_series(flows, "2021-07-26", "2021-08-25", title="SEED drought event")

In [ ]:
# Metrics of every model against USGS observations during the SEED drought event
drought_metrics = period_metrics(flows, "2021-07-26", "2021-08-25")
plot_metrics_bars(drought_metrics, title="SEED drought event: models vs. USGS observations")

<h4 style="background-color:rgba(184,34,61,0.18); padding:8px 12px; border-left: 5px solid #E04A63; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 SEED Flood Event (2006-12-29 to 2007-01-12)
</h4>


In [ ]:
# Daily streamflow during the SEED flood event
plot_period_series(flows, "2006-12-29", "2007-01-12", title="SEED flood event")

In [ ]:
# Metrics of every model against USGS observations during the SEED flood event
flood_metrics = period_metrics(flows[["Observed", "NWM", "NextGen (AORC)", "NextGen (IMERG)"]], "2006-12-29", "2007-01-12")
plot_metrics_bars(flood_metrics, title="SEED flood event: models vs. USGS observations")

<div style="background:#9E1B32; padding:14px 18px; border-radius:8px; text-align:center; margin-top:10px;">
  <a href="03.model_run.ipynb" target="_blank" style="color:#ffffff; font-size:18px; font-weight:700; text-decoration:none;">Finish - Questions?</a>
</div>